# 🇰🇪 DataSprint 2026 — Financial Status Prediction
### Strathmore Data Community × iLab Africa
**Dataset:** 2024 FinAccess Household Survey — 20,871 Kenyan Adults  
**Task:** Predict whether a Kenyan adult's financial situation has *Improved*, *Stayed the same*, or *Worsened*  
**Primary Metric:** Weighted F1-Score  
**Models:** Decision Tree (baseline) + Random Forest (primary)


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Colour palette used throughout
STATUS_ORDER  = ['Worsened', 'Stayed the same', 'Improved']
STATUS_COLORS = ['#C0392B', '#F39C12', '#27AE60']

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})
print("✅ Libraries loaded")


## 2. Load & Inspect Data

In [ ]:
# Upload finaccess2024_datasprint.csv to Colab first, then run this cell
df = pd.read_csv('finaccess2024_datasprint.csv')

print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nTarget distribution:")
print(df['financial_status'].value_counts())
df.head(3)


## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Chart 1 — Target class distribution
fig, ax = plt.subplots(figsize=(8, 5))
counts = df['financial_status'].value_counts().reindex(STATUS_ORDER)
bars = ax.bar(STATUS_ORDER, counts.values, color=STATUS_COLORS, width=0.5, edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Financial Status of Kenyan Adults (2024 FinAccess Survey)', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Respondents')
ax.set_ylim(0, 14000)
plt.tight_layout()
plt.show()
print("Key insight: Over half of Kenyans (52.6%) reported their finances WORSENED.")


In [ ]:
# Chart 2 — Financial shock vs outcome
fig, ax = plt.subplots(figsize=(8, 5))
shock = df.groupby(['experienced_shock', 'financial_status']).size().unstack().reindex(columns=STATUS_ORDER)
shock_pct = shock.div(shock.sum(axis=1), axis=0) * 100
shock_pct.plot(kind='bar', ax=ax, color=STATUS_COLORS, width=0.6, edgecolor='white')
ax.set_title('Financial Outcome by Shock Experience', fontsize=13, fontweight='bold')
ax.set_xticklabels(['No Shock', 'Experienced Shock'], rotation=0, fontsize=11)
ax.set_ylabel('Percentage (%)')
ax.legend(STATUS_ORDER, title='Status')
ax.set_ylim(0, 80)
for c in ax.containers:
    ax.bar_label(c, fmt='%.1f%%', fontsize=8, padding=2)
plt.tight_layout()
plt.show()
print("Key insight: People who experienced a financial shock were much more likely to report Worsened status.")


In [ ]:
# Chart 3 — Monthly income distribution by status
fig, ax = plt.subplots(figsize=(9, 5))
for status, color in zip(STATUS_ORDER, STATUS_COLORS):
    data = df[df['financial_status'] == status]['monthly_income']
    ax.hist(data[data <= 30000], bins=30, alpha=0.6, color=color, label=status, edgecolor='white')
ax.set_title('Monthly Income Distribution by Financial Status', fontsize=13, fontweight='bold')
ax.set_xlabel('Monthly Income (KES) — capped at 30,000 for clarity')
ax.set_ylabel('Number of Respondents')
ax.legend()
plt.tight_layout()
plt.show()

# Print median incomes per group
for status in STATUS_ORDER:
    med = df[df['financial_status'] == status]['monthly_income'].median()
    print(f"  Median income — {status}: KES {med:,.0f}")


In [ ]:
# Chart 4 — Education level vs financial status
fig, ax = plt.subplots(figsize=(9, 5))
edu = df.groupby(['education_level', 'financial_status']).size().unstack().reindex(columns=STATUS_ORDER)
edu_pct = edu.div(edu.sum(axis=1), axis=0) * 100
edu_pct.plot(kind='bar', ax=ax, color=STATUS_COLORS, width=0.7, edgecolor='white')
ax.set_title('Financial Status by Education Level', fontsize=13, fontweight='bold')
ax.set_xlabel('Education Level  (1 = None  →  5 = University)')
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(STATUS_ORDER, title='Status')
plt.tight_layout()
plt.show()
print("Key insight: Higher education is associated with better financial outcomes.")


In [ ]:
# Chart 5 — Financial health indicators
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
health_cols = [('nfhi_11', 'Food Secure\n(past 12 months)'),
               ('nfhi_12', 'Managed Non-Food\nSpending'),
               ('nfhi_13', 'No Debt Stress\n(past 3 months)')]
for ax, (col, label) in zip(axes, health_cols):
    data = df.groupby([col, 'financial_status']).size().unstack().reindex(columns=STATUS_ORDER)
    pct  = data.div(data.sum(axis=1), axis=0) * 100
    pct.plot(kind='bar', ax=ax, color=STATUS_COLORS, width=0.6, edgecolor='white', legend=False)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xticklabels(['No', 'Yes'], rotation=0)
    ax.set_ylabel('Percentage (%)')
axes[2].legend(STATUS_ORDER, title='Status', fontsize=8)
fig.suptitle('Financial Health Indicators vs Financial Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Key insight: Food security and being debt-free are strongly linked to financial improvement.")
